# Task 2.1: Implementación, calibración y sensibilidad

- **INCISO A:** 
Método de Runge-Kutta de cuarto orden (RK4) para simular el avance diario de la epidemia. Extrae los valores en los días 7, 14, 21 y 28 (semanas 1 a 4) para compararlos con los datos observados y calcular la Suma de Cuadrados del Error (SCE).

In [ ]:
import numpy as np

# Diapositiva 4: estructura y supuestos del modelo SIR (S, I, R, población cerrada, beta y gamma).
# Diapositiva 11: reto de calibración con datos reales y datos observados en semanas 1 a 4.
# Definición de parámetros iniciales (Task 1.1)
N = 500000
I0 = 800
R0 = 200
S0 = N - I0 - R0
beta = 0.4
gamma = 1/7

# Datos observados en las semanas 1, 2, 3 y 4 (días 7, 14, 21 y 28)
I_obs = np.array([1850, 4200, 8900, 16400])
t_obs = np.array([7, 14, 21, 28])

# Diapositiva 4: sistema de ecuaciones diferenciales del modelo SIR.
# Diapositiva 5: R0 = beta/gamma y criterio de crecimiento/extinción.
# Definición del sistema de EDOs
def sir_derivadas(t, y, N, beta, gamma):
    S, I, R = y
    dSdt = -beta * (S * I) / N
    dIdt = beta * (S * I) / N - gamma * I
    dRdt = gamma * I
    return np.array([dSdt, dIdt, dRdt])

# Diapositiva 7: validación del modelo mediante comparación y ajuste cuantitativo.
# Diapositiva 12: análisis de sensibilidad del parámetro beta.
# Implementación del método RK4
def rk4_paso(f, t, y, h, N, beta, gamma):
    k1 = f(t, y, N, beta, gamma)
    k2 = f(t + h/2, y + (h/2)*k1, N, beta, gamma)
    k3 = f(t + h/2, y + (h/2)*k2, N, beta, gamma)
    k4 = f(t + h, y + h*k3, N, beta, gamma)
    return y + (h/6) * (k1 + 2*k2 + 2*k3 + k4)

# Diapositiva 11: simulación del avance diario de la epidemia con datos reales.
# Simulación
dias_simulacion = 28
h = 1 # Paso de tiempo de 1 día
t = 0
y = np.array([S0, I0, R0])

# Listas para almacenar resultados
I_predicho = []

for dia in range(1, dias_simulacion + 1):
    y = rk4_paso(sir_derivadas, t, y, h, N, beta, gamma)
    t += h
    # Guardar predicciones en las semanas exactas (días 7, 14, 21, 28)
    if dia in t_obs:
        I_predicho.append(y[1])

I_predicho = np.array(I_predicho)

# Diapositiva 7: cálculo del error para validar la calibración del modelo.
# Cálculo de la Suma de Cuadrados del Error (SCE)
SCE = np.sum((I_obs - I_predicho)**2)

print("--- Resultados de la Simulación Base ---")
for i, semana in enumerate(range(1, 5)):
    print(f"Semana {semana}: I_obs = {I_obs[i]:.0f} | I_predicho = {I_predicho[i]:.2f}")
print(f"Suma de Cuadrados del Error (SCE) = {SCE:.2f}")

--- Resultados de la Simulación Base ---
Semana 1: I_obs = 1850 | I_predicho = 4752.92
Semana 2: I_obs = 4200 | I_predicho = 25755.18
Semana 3: I_obs = 8900 | I_predicho = 92838.09
Semana 4: I_obs = 16400 | I_predicho = 137455.70
Suma de Cuadrados del Error (SCE) = 22173137013.19


**Inciso C:** variará el parámetro $\beta$ en $\pm 20\%$, simulará un brote completo y reportará las métricas solicitadas.

In [ ]:
# Diapositiva 12: análisis de sensibilidad variando beta en ±20% para evaluar robustez del modelo.
# Parámetros para el análisis de sensibilidad
betas_sensibilidad = [0.4 * 0.8, 0.4, 0.4 * 1.2] # [0.32, 0.40, 0.48]
dias_totales = 150 
resultados_sensibilidad = []

print("--- Análisis de Sensibilidad (Variación de Beta) ---")
for beta_test in betas_sensibilidad:
    t = 0
    y = np.array([S0, I0, R0])
    I_trayectoria = [I0]
    R_trayectoria = [R0]
    
    for dia in range(1, dias_totales + 1):
        y = rk4_paso(sir_derivadas, t, y, h, N, beta_test, gamma)
        t += h
        I_trayectoria.append(y[1])
        R_trayectoria.append(y[2])
        
    pico_I = max(I_trayectoria)
    tiempo_pico = I_trayectoria.index(pico_I)
    tamaño_final_epidemia = y[1] + y[2] # Infecciones acumuladas al final
    
    resultados_sensibilidad.append(pico_I)
    
    print(f"\nBeta = {beta_test:.2f}:")
    print(f"  Pico de I(t) = {pico_I:.0f} personas")
    print(f"  Tiempo al pico = {tiempo_pico} días")
    print(f"  Tamaño final de la epidemia = {tamaño_final_epidemia:.0f} personas")

# Diapositiva 5: criterio de umbral y análisis de impacto del aumento de R0.
# Cálculo de la variación porcentual del pico
variacion_positiva_beta = ((resultados_sensibilidad[2] - resultados_sensibilidad[1]) / resultados_sensibilidad[1]) * 100
print(f"\nVariación del pico al aumentar beta un 20%: +{variacion_positiva_beta:.2f}%")

--- Análisis de Sensibilidad (Variación de Beta) ---

Beta = 0.32:
  Pico de I(t) = 97004 personas
  Tiempo al pico = 37 días
  Tamaño final de la epidemia = 425891 personas

Beta = 0.40:
  Pico de I(t) = 137456 personas
  Tiempo al pico = 28 días
  Tamaño final de la epidemia = 462529 personas

Beta = 0.48:
  Pico de I(t) = 170935 personas
  Tiempo al pico = 22 días
  Tamaño final de la epidemia = 480170 personas

Variación del pico al aumentar beta un 20%: +24.36%
